# Rasters, Arrays, and rasterio

Part 2 has so far used vector data: points, lines, and polygons

This session introduces the raster data model. A raster is a grid with one value per cell, and rasterio returns that grid as a numpy array, the structure introduced way in week 3. 

The session's raster is `portage_dem.tif`, a digital elevation model of Portage County, Ohio, with a 2 km buffer, at about 26.6 m resolution, derived from Copernicus satellite elevation data.

By the end of this session you will be able to:

- Open a raster with rasterio and read its **profile** (CRS, shape, dtype, nodata) BEFORE reading any pixel values
- Index and slice a raster's array
- Perform a windowed read to extract part of a raster without loading the whole grid
- Plot a raster with matplotlib and a correctly scaled colorbar
- Identify nodata in an array, mask it before computing statistics, and recognize that a file's metadata may not describe every sentinel value present

In [ ]:
# Run this cell first (takes ~1 minute). Colab already includes geopandas.
%pip install -q libpysal esda splot mapclassify contextily
%pip install -q rasterio rioxarray rasterstats

## What a raster is

Before we start: find a buddy, work through the following questions and be prepared to discuss.

1. **What is a raster?** Describe its structure in a sentence without using the word "image."
2. **What does a raster store that a vector layer does not, and what shortcomings or compromises does it have?** A vector layer has rows, geometries, and attribute columns. Which of those does a raster have, and which does it lack?
3. **What makes a grid of numbers a map rather than a picture?** A photograph is also a grid of values. What additional information must a raster include so that a cell can be placed on the ground?
4. **What does resolution limit?** If every cell is 30 m across, what kinds of real features can and cannot be represented?
5. **What value does a cell take where there is no measurement?** A raster is rectangular; a county is not. What fills the cells outside the county, and why does that matter for a mean or a count?
6. **Can one raster hold more than one grid?** A DEM holds elevation. What would a satellite image need to hold?



### Answers

**1. What is a raster?** A rectangular grid of equally sized cells, with one value per cell and no features. A cell is also sometimes called a pixel. The value may vary continuously across space (elevation, temperature, reflectance) or be a category code (a land-cover class, a flag)

**2. What it stores and what shortcomings/compromises** A raster has values and position, and nothing else: no rows that correspond to things, no geometry per feature, no attribute table. It represents a surface rather than a set of objects. It gains the ability to describe something that varies everywhere, which a polygon layer cannot do without drawing a boundary around every change. It loses the identity of individual features: a raster of elevation does not know where one hill ends and the next begins. NO TOPOLOGY

**3. What makes it a map?** Six properties, which together form the raster's metadata:

- **Shape**: the number of rows and columns. Row 0 is the top of the grid, column 0 is the left edge
- **Resolution**: the size of one cell on the ground, in the units of the CRS
- **Extent**: the real-world rectangle the grid covers, stated as its bounds (left, bottom, right, top)
- **CRS**: the coordinate reference system that provides the resolution and extent their units
- **Transform**: the rule that converts a cell position `(row, col)` into a ground coordinate `(x, y)` and back, determined by the top-left corner coordinate and the cell size. This separates a raster from a photograph.
- **Nodata**: a reserved value meaning "no measurement in this cell."

**4. What resolution limits.** A 30 m cell reports one value for the whole 30 m by 30 m square. A feature narrower than a cell, such as a stream channel or a road, is either absorbed into the surrounding value or represented by a cell much wider than itself. Resolution sets the smallest thing a raster can show, and no processing can recover detail that was never recorded.

**5. Cells without a measurement.** They hold the nodata value, a number reserved for the purpose (often `-9999` for elevation) that fills the corners outside the study area and any gaps inside it. It is stored in the grid like any other number, so an unmasked mean or count includes it ***as if it were a measurement***. 

**6. Bands.** A raster *may* hold several bands, each a separate grid with the same shape and extent. A DEM has one band. A satellite image has one band per wavelength. In this course rasters are (mostly) stored as **GeoTIFF** (`.tif`) files, an image format with the metadata above written into its header, which is what distinguishes it from an ordinary TIFF.

In Python, a band is a two-dimensional numpy array, and  metadata is read from the file's header. 

### Let's build a small example raster

In [ ]:
import numpy as np
from rasterio.transform import from_origin ### Let's look at this too

# A raster with 3 rows and 4 columns, one value per cell. Row 0 is the top edge.
grid = np.array([[10, 11, 12, 13],
                 [20, 21, 22, 23],
                 [30, 31, 32, 33]])
print("shape (rows, cols):", grid.shape)
print("top-left cell:", grid[0, 0], "   bottom-right cell:", grid[2, 3])

# The transform ties the grid to the ground: top-left corner at x=500000, y=4580000 (meters),
# cells 30 m wide and 30 m tall. Only these numbers make the grid a map rather than a picture.
transform = from_origin(500000, 4580000, 30, 30)
print("corner of (col 0, row 0):", transform * (0, 0))
print("corner of (col 4, row 3):", transform * (4, 3))   # one past the last column and row: the bottom-right corner
print("extent (left, bottom, right, top):", 500000, 4580000 - 3 * 30, 500000 + 4 * 30, 4580000)

**Expected result:** 

- shape `(3, 4)`
- top-left cell `10`
- bottom-right cell `33`

The transform maps `(col 0, row 0)` to `(500000.0, 4580000.0)`, the top-left corner, and `(col 4, row 3)` to `(500120.0, 4579910.0)`, the bottom-right corner: moving right adds 30 m per column to `x`, and moving down subtracts 30 m per row from `y`.

Note the order: numpy indexes `grid[row, col]`, but the transform takes `(col, row)`, because `x` corresponds to column and `y` to row. 
 
The extent is 120 m wide and 90 m tall, which is the shape multiplied by the resolution. 

## Opening a raster: but looking at the profile first

A GeoTIFF is more than a grid of numbers. `rasterio.open()` returns a **profile**: the raster's CRS, its shape (rows by columns), its data type, its nodata value, and the affine transform that maps grid position to real-world coordinates. Read the profile before reading any value. It plays the role that `.crs` and `.head()` play for a GeoDataFrame.


### Why might we want to read information ABOUT the image BEFORE we actually read the image?

In [ ]:
DATA_BASE_P2 = "https://raw.githubusercontent.com/pjbitterman/KSU_x9076/main/data/part2"

# Local path fallback (uncomment if running outside Colab without internet access):
# DATA_BASE_P2 = "../data/part2"

import rasterio

dem_path = f"{DATA_BASE_P2}/portage_dem.tif"

with rasterio.open(dem_path) as src: ### Let's break this line down. How does "with" work? 
    print("CRS:", src.crs)
    print("shape (rows, cols):", src.shape)
    print("dtype:", src.dtypes)
    print("nodata:", src.nodata)
    print("pixel size (m):", src.res)
    print("bounds:", src.bounds)

### What did we get?


### About "with"

- `with` opens a resource (a file, a raster, a cursor), gives it a name for the indented block, and closes it automatically when the block ends, even if an error occurs. You never call close() yourself.

- The name only works inside the block. Read what you need into a variable there, such as arr = src.read(1), and use that variable afterward; the array outlives the file.

- It is not a loop or a condition, despite the indentation. It "brackets" how long you hold onto a file (or something else), which is why every rasterio.open, open(), and arcpy.da.SearchCursor in this course sits inside one.

### Exercise

Open `portage_dem.tif` again and print `src.width`, `src.height`, and `src.count` (number of bands) directly, without `.shape`. Confirm that `src.width == src.shape[1]` and `src.height == src.shape[0]`. Which of the two is rows and which is columns?

In [ ]:
## Try it here



## Array indexing

The early "intro weeks" covered indexing and slicing lists: `data[0]`, `data[2:5]`, and loops over values. 

A raster's pixel values reside in the same kind of structure. 

`.read(1)` returns band 1 as a two-dimensional numpy array, and every list-indexing operation applies, with two axes (`[row, col]`) instead of one.

**Before running.** `arr[0, 0]` is the raster's northwest-most cell. Rasters are rectangular even when the area of interest is not, so that corner likely falls outside Portage County's irregular boundary. Predict whether it holds an elevation value or nodata.

In [ ]:
with rasterio.open(dem_path) as src:
    arr = src.read(1)

print("array shape:", arr.shape, "dtype:", arr.dtype)
print("top-left corner, arr[0, 0]:", arr[0, 0])
print("center pixel, arr[765, 694]:", arr[765, 694])
print("a 3x3 neighborhood around the center:")
print(arr[764:767, 693:696])

**Result?** 

**Exercise.** Slice `arr[0:50, 0:50]`, the top-left 50-by-50 corner, and count how many of those 2,500 values equal `-9999.0` with boolean indexing: `(subset == -9999).sum()`. 

Is it all 2,500, none, or some fraction? What does the count say about where the county boundary crosses that corner of the file?

## Windowed reads

`arr = src.read(1)` loaded all 2.1 million pixels into memory. That is acceptable for a file this size, but DEMs and satellite imagery can run to gigabytes, beyond what fits in memory. `rasterio.windows.Window` reads only the requested rectangle directly from disk.

### Let's try.

Break it down step by step

In [ ]:
from rasterio.windows import Window

# Define a 300x300 pixel window starting at column 600, row 600. The window is a rectangle in pixel coordinates, not map coordinates.
window = Window(col_off=600, row_off=600, width=300, height=300) ## Note the difference in capitalization of "Window" vs "window"

# Read the window from the DEM and get its own transform. The transform is different from the full DEM's transform because the window's top-left corner is not at the same map coordinates as the full DEM's top-left corner.
with rasterio.open(dem_path) as src:
    sub = src.read(1, window=window)
    window_transform = src.window_transform(window)

print("window shape:", sub.shape)
print("window min/max elevation:", sub.min(), sub.max())
print("any nodata in this window?", (sub == -9999.0).any())
print("window's own affine transform:", window_transform)

**Expected result:** shape `(300, 300)`, elevation from about `304.8` to `396.8` m, and no nodata, because this window lies inside the county interior away from the rectangular corners. 

Note the window's own transform: its top-left origin has moved from the full raster's `(464899.32, 4579530.44)` to a point 600 pixels further in. `window_transform` performs that bookkeeping, so a windowed read's pixels still map to their correct real-world locations.

**Exercise.** Build a second `Window` covering the bottom-right 200-by-200 corner (`col_off=1388 - 200`, `row_off=1530 - 200`, `width=200`, `height=200`) and read it. 

Does that corner contain nodata, elevation, or both, and is that consistent with the top-left corner's result?

In [ ]:
# Try it here


## Plotting the DEM

A plot makes terrain legible in a way a printed array cannot. 

matplotlib's `imshow()` renders the array directly, and a colorbar maps pixel color back to meters.

**Before running.** Plotting `arr` directly with `imshow()`, nodata included, uses one continuous color scale across the entire value range, from `-9999` to `427.9`. 

Predict the effect on the terrain detail. For example, what's the 261 to 428 m elevation range going to look like?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


dem_masked = np.ma.masked_equal(arr, -9999.0) # mask the nodata value (-9999.0) in the DEM array

fig, axes = plt.subplots(1, 2, figsize=(13, 6)) # create a figure with 1 row and 2 columns of subplots

im0 = axes[0].imshow(arr, cmap="terrain") # display the unmasked DEM array in the first subplot
axes[0].set_title("Unmasked -- nodata included") # set the title for the first subplot
fig.colorbar(im0, ax=axes[0], label="raw pixel value (m)") # add a colorbar to the first subplot with label "raw pixel value (m)"

im1 = axes[1].imshow(dem_masked, cmap="terrain") # display the masked DEM array in the second subplot. cmap is set to "terrain" for a color map that represents elevation
axes[1].set_title("Masked -- nodata excluded") # set the title for the second subplot
fig.colorbar(im1, ax=axes[1], label="elevation (m)") # add a colorbar to the second subplot with label "elevation (m)"

plt.tight_layout() # adjust the layout of the subplots to prevent overlap
plt.show() # display the figure with the subplots

**Result:** 

- The left panel's colorbar extends to `-9999`, so the entire 261 to 428 m elevation range is compressed into a thin band at the top of the scale and the terrain appears nearly uniform, while the nodata corners occupy most of the visual range. 
- The right panel, built from `np.ma.masked_equal()`, excludes nodata from the color scale and the relief becomes visible.

**Exercise.** Re-plot the masked DEM with a different colormap, such as `"viridis"` or `"cividis"` in place of `"terrain"`, and add a `plt.title()` and axis labels. Which colormap makes the elevation gradient easier to read?

In [ ]:
# Try it here


## Nodata, two cases

A raster's `.nodata` attribute declares one sentinel value. ***Wait, what's a sentinel value?***

*a number chosen because it could never be a real measurement, such as -9999 for elevation, stored in the data to mean "no value here" rather than to be used as a value.*

It does NOTE guarantee that every invalid value in the file is accounted for. 

Let's look at two examples
- First, the DEM's `-9999`, a clean and well-declared case
- Second, another raster from a satellite bloom-surveillance product, where the declared nodata is not the complete picture

### Let's start with computing the mean

In [ ]:
naive_mean = arr.mean()
valid = arr[arr != -9999.0]
honest_mean = valid.mean()

print(f"naive mean (nodata included): {naive_mean:.1f} m")
print(f"honest mean (nodata excluded): {honest_mean:.1f} m")
print(f"share of pixels that are nodata: {(arr == -9999.0).sum() / arr.size:.1%}")

**Expected result:** raw mean `191.3` m, masked mean `336.5` m, a difference of **145 m** from a nodata share of 1.4%. 

`-9999` is not a small perturbation; it is an extreme negative outlier, and a small fraction of pixels containing it shift the mean substantially. The arithmetic is correct and the result is meaningless until nodata is masked.

In [ ]:
# A second worked example: a boolean condition rather than a mean
naive_low_count = (arr < 300).sum()
honest_low_count = (valid < 300).sum()

print(f"naive count of 'pixels below 300 m' (nodata included): {naive_low_count:,}")
print(f"honest count (nodata excluded): {honest_low_count:,}")

**Expected result:** `244,509` raw versus `214,685` masked. E

Each of the raster's 29,824 nodata pixels is `-9999`, which satisfies "below 300," so an unmasked boolean count includes all of them alongside valid valley-floor readings. 

The correction is the same in every case: filter to `valid = arr[arr != nodata]`, or mask, before doing any math/statistics

## A second raster: an undeclared sentinel

The DEM's nodata is the simple case: one sentinel, declared in the metadata, removed with one line. The second raster is harder. 

`champlain_cyano_2016.tif` is a harmful-algal-bloom surveillance product for Lake Champlain, a 7-day-maximum Cyanobacteria Index composite in the family of satellite products NOAA's Cyanobacteria Assessment Network (CyAN) produces for lakes nationwide.

**Before running.** This file declares `nodata=255`, the same pattern as the DEM's `-9999`. Once those declared-nodata pixels are masked, does the remainder look like a continuously varying satellite index? Consider why this file warrants its own worked example.

In [ ]:
cyano_path = f"{DATA_BASE_P2}/champlain_cyano_2016.tif" # path

with rasterio.open(cyano_path) as src: # read the raster file at cyano_path using rasterio and assign it to the variable src
    cyano = src.read(1)
    cyano_crs = src.crs
    cyano_nodata = src.nodata

print("CRS:", cyano_crs)
print("shape:", cyano.shape, "dtype:", cyano.dtype)
print("declared nodata:", cyano_nodata)

# Plot the raw unmasked raster first
plt.figure(figsize=(6, 6))
plt.imshow(cyano, cmap="viridis")
plt.title("Raw Unmasked Cyanobacteria Raster")
plt.colorbar(label="Raw Pixel Value")
plt.show()


In [ ]:
# Next, mask out the declared nodata values and look at the range of valid pixel values
cyano_valid = cyano[cyano != cyano_nodata]
print(f"share masked as declared nodata: {(cyano == cyano_nodata).sum() / cyano.size:.1%}")
print(f"'valid' pixel value range: {cyano_valid.min()}-{cyano_valid.max()}")

# To plot, we must use a masked 2D array instead of the flattened 1D cyano_valid array
cyano_valid_2d = np.ma.masked_equal(cyano, cyano_nodata)

# Set up a single plot matching the previous matplotlib structure
plt.figure(figsize=(6, 6))
im = plt.imshow(cyano_valid_2d, cmap="viridis")
plt.title("Masked Cyanobacteria Index (Excluding 255)")
plt.colorbar(im, label="Cyanobacteria Digital Number (60 - 252)")
plt.show()

# Next, let's find the most common 'valid' value and its share of all 'valid' pixels
values, counts = np.unique(cyano_valid, return_counts=True)
top_value = values[np.argmax(counts)]
top_share = counts.max() / cyano_valid.size
print(f"most common 'valid' value: {top_value}, {top_share:.2%} of all 'valid' pixels")

**Expected result:** masking the declared `nodata=255` removes only 7.8% of pixels; the remaining 92.2% range from `60` to `252`. The distribution within that range is the finding: **99.34%** of the remaining pixels hold the single value `252`.

A continuously varying index does not behave that way. Converted through NOAA's published CyAN digital-number formula (`CIcyano = 10^(DN * 0.011714 - 4.1870866)`), a DN of `252` sits near the top of the detectable-bloom scale, which cannot describe 99% of a lake's surface at once. 

The likely explanation is that `252` is a second, undeclared sentinel, such as a land-adjacency or low-confidence flag, that the file's `nodata` metadata does not mention. Masking the declared value was the first check, not the last.

In [ ]:
# Excluding both the declared nodata (255) AND the suspect flag value (252)
cyano_real = cyano_valid[cyano_valid != 252]

print(f"pixels left after excluding both sentinels: {cyano_real.size:,} of {cyano.size:,} ({cyano_real.size / cyano.size:.2%} of the whole raster)")
print(f"remaining value range: {cyano_real.min()}-{cyano_real.max()}")
print(f"mean of the 252-inflated 'valid' set: {cyano_valid.mean():.1f}")
print(f"mean after also excluding 252: {cyano_real.mean():.1f}")

**Expected result:** excluding `252` alongside `255` leaves `1,171` of the raster's `192,468` pixels, about 0.6% of the file (0.66% of the pixels that passed the declared-nodata filter). That is the portion of the lake where this composite reports a distinct index reading, from `60` to `248`. The mean falls from `251.7` to `205.1` once `252` no longer dominates it. 

A declared `nodata` value is a starting assumption, and the only way to detect a second sentinel is to examine the distribution of what remains.

**Exercise.** Plot a masked version of `cyano` that excludes all values greater than or equal to `252` 
(`np.ma.masked_where(cyano >= 255)` extends `masked_equal` to two conditions) with `imshow()` and a colorbar. 

Where on the lake do the remaining pixels cluster: spread across the lake, or concentrated in one area? There is no single expected answer; describe what is there.

In [ ]:
## Try it here


## Summary

A raster's declared nodata value is where the validity check STARTS. 

The DEM's `-9999` was the simple case, one documented sentinel removed with `arr != nodata`

`champlain_cyano_2016.tif` was the realistic case: the declared nodata was correct but incomplete, and a second sentinel sat inside the nominally valid range.

## Wrap-up and next steps

**Practice** (no submission):

1. Open `portage_dem.tif` again and compute the masked standard deviation of elevation with `valid.std()`. Given the masked mean (`336.5` m) and range (`261` to `428` m), is the standard deviation proportionate to that range?
2. Read a `Window` centered on the raster (`col_off=694 - 150, row_off=765 - 150, width=300, height=300`) and compare its minimum and maximum elevation to the earlier window's.
3. On `champlain_cyano_2016.tif`, count the pixels in the `1` to `59` range, below the file's observed valid minimum of 60. Is the count zero?


**Due Wednesday** Lab 4 ("Vectors, Open Source: Chesapeake Stations")

**Next session:** raster algebra, slope, hillshade, and zonal statistics on this DEM

*Adapted from Qiusheng Wu, "Introduction to GIS Programming" (geog-312), CC BY 4.0.*